# Post Pandemic Regime Shifts in Labor Market: Feature Engineering

## Purpose

## Research Context

## Imports and Configuration

In [62]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [63]:
%matplotlib inline
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from scipy import stats
from scipy.stats import t as t_dist
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams["figure.figsize"] = (14, 8)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 50)
sns.set_theme(style="whitegrid")

In [64]:
from regime_shift.config import (
    MERGED_DATA_PATH,
    FEATURE_ENGINEERED_DATA_PATH,
    REPORTS_DIR,
    FIGURES_DIR,
    REGIME_BREAKS,
    QUANDT_BREAKPOINT,
    LABOR_MARKET_VARIABLES,
    WAGE_VARIABLES,
    INFLATION_VARIABLES,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## Load Dataset

In [65]:
df = pd.read_csv(MERGED_DATA_PATH)

df = df.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(df.shape)
df.columns.tolist()

(315, 136)


['date',
 'job_openings_level',
 'hires_rate',
 'hires_level',
 'quits_rate',
 'quits_level',
 'total_separations_rate',
 'total_separations_level',
 'layoffs_discharges_level',
 'layoffs_discharges_rate',
 'unemployment_rate',
 'unemployment_level',
 'u6_rate',
 'prime_age_lfpr',
 'lfpr',
 'epop_ratio',
 'prime_age_urate',
 'avg_weeks_unemployed',
 'payrolls_nonfarm',
 'payrolls_private',
 'payrolls_manufacturing',
 'payrolls_services',
 'payrolls_construction',
 'ahe_private',
 'awe_private',
 'awh_private',
 'awe_manufacturing',
 'ahe_manufacturing',
 'eci_total',
 'eci_wages',
 'cpi_all',
 'cpi_core',
 'cpi_less_shelter',
 'cpi_services_less_rent',
 'cpi_services_less_energy',
 'cpi_shelter',
 'cpi_medical',
 'cpi_food',
 'pce_price',
 'pce_core',
 'pce_trimmed_12m',
 'pce_trimmed_1m',
 'pce_services',
 'saving_rate',
 'real_dpi',
 'real_pi',
 'real_pi_less_transfers',
 'retail_advance',
 'real_retail',
 'ip_total',
 'ip_manufacturing',
 'capacity_util',
 'housing_starts',
 'buildi

## Regime Definition

In [66]:
pre_2020_cutoff = pd.Timestamp("2020-01-01")
recovery_threshold = pd.Timestamp("2021-06-01")
quandt_breakpoint = pd.Timestamp("2021-09-01")

df["regime_pre2020"] = (df["date"] < pre_2020_cutoff).astype(int)
df["regime_pandemic"] = ((df["date"] >= pre_2020_cutoff) & (df["date"] < recovery_threshold)).astype(int)
df["regime_post2021"] = (df["date"] >= recovery_threshold).astype(int)
df["regime_post_quandt"] = (df["date"] >= quandt_breakpoint).astype(int)

df["regime"] = "Pre-2020"
df.loc[df["date"] >= pre_2020_cutoff, "regime"] = "Pandemic Shock (2020)"
df.loc[df["date"] >= recovery_threshold, "regime"] = "Post-Pandemic (2021+)"

print("Regime Distribution:\n", df["regime"].value_counts())

Regime Distribution:
 regime
Pre-2020                 240
Post-Pandemic (2021+)     58
Pandemic Shock (2020)     17
Name: count, dtype: int64
